In [1]:
from pathlib import Path
import csv
import math
import struct

import dpkt
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import DBSCAN, KMeans, SpectralClustering
import hdbscan


## Filtering, subsampling, and clustering

In [2]:
def prepare_points(points, min_distance, max_distance, keep_fraction, seed):
    distances = np.linalg.norm(points, axis=1)
    filtered = points[(distances >= min_distance) & (distances <= max_distance)]
    if len(filtered) and keep_fraction < 1.0:
        keep_count = max(1, int(np.ceil(len(filtered) * keep_fraction)))
        rng = np.random.default_rng(seed)
        indices = np.sort(rng.choice(len(filtered), size=keep_count, replace=False))
        filtered = filtered[indices]
    return filtered


def cluster_points(points, method, cluster_count, dbscan_eps, min_samples, hdbscan_min_cluster_size, spectral_neighbors):
    point_count = len(points)
    if point_count == 0:
        return np.empty(0, dtype=int)
    if method == 'K-means':
        return KMeans(n_clusters=min(cluster_count, point_count), n_init=10, random_state=42).fit_predict(points)
    if method == 'Spectral':
        if point_count < 2:
            return np.zeros(point_count, dtype=int)
        count = min(cluster_count, point_count - 1)
        neighbors = min(spectral_neighbors, point_count - 1)
        return SpectralClustering(n_clusters=count, affinity='nearest_neighbors', n_neighbors=neighbors, assign_labels='kmeans', random_state=42).fit_predict(points)
    if method == 'DBSCAN':
        return DBSCAN(eps=dbscan_eps, min_samples=min_samples).fit_predict(points)
    if method == 'HDBSCAN':
        if point_count < 2:
            return np.full(point_count, -1, dtype=int)
        return hdbscan.HDBSCAN(min_cluster_size=min(hdbscan_min_cluster_size, point_count), min_samples=min_samples).fit_predict(points)
    raise ValueError(f'Unknown clustering method: {method}')


def cluster_frame(points, method, settings, seed):
    selected = prepare_points(points, settings['min_distance'], settings['max_distance'], settings['keep_fraction'], seed)
    labels = cluster_points(selected, method, settings['cluster_count'], settings['dbscan_eps'], settings['min_samples'], settings['hdbscan_min_cluster_size'], settings['spectral_neighbors'])
    cluster_labels = set(labels.tolist()) - {-1}
    return {
        'points': selected,
        'labels': labels,
        'decoded_count': len(points),
        'used_count': len(selected),
        'cluster_count': len(cluster_labels),
        'noise_count': int(np.count_nonzero(labels == -1)),
    }

## Open file

In [3]:
PCAP_PATH = Path("../data/myself in my room dancing.pcap")
if not PCAP_PATH.exists():
    PCAP_PATH = Path("data/driveway.pcap")

CALIBRATION_PATH = Path("../Lidar setup/JT128_angle_correction.csv")
if not CALIBRATION_PATH.exists():
    CALIBRATION_PATH = Path("Lidar setup/JT128_angle_correction.csv")

MAX_FRAMES = 40
if isinstance(MAX_FRAMES, bool) or not isinstance(MAX_FRAMES, int) or MAX_FRAMES < 1:
    raise ValueError("MAX_FRAMES must be a positive integer.")


def load_calibration(path):
    calibration = {}
    with path.open("r", encoding="utf-8", errors="ignore", newline="") as calibration_file:
        for row in csv.DictReader(calibration_file):
            try:
                channel = int(row["Channel"])
                elevation = float(row["Elevation"])
                azimuth = float(row["Azimuth"])
            except (KeyError, TypeError, ValueError):
                continue
            calibration[channel] = (elevation, azimuth)

    if sorted(calibration) != list(range(1, 129)):
        raise ValueError(f"Expected 128 JT128 calibration rows in {path}.")

    elevations = np.array([calibration[channel][0] for channel in range(1, 129)], dtype=float)
    azimuth_offsets = np.array([calibration[channel][1] for channel in range(1, 129)], dtype=float)
    return elevations, azimuth_offsets


VERTICAL_ANGLES_DEG, AZIMUTH_OFFSETS_DEG = load_calibration(CALIBRATION_PATH)


def decode_packet(payload):
    if len(payload) < 12 or payload[:4] != b"\xee\xff\x01\x04":
        return []

    laser_count, block_count, _, distance_unit_mm, _, status = payload[6:12]
    if laser_count != len(VERTICAL_ANGLES_DEG):
        raise ValueError(f"Packet has {laser_count} channels, but the calibration has {len(VERTICAL_ANGLES_DEG)}.")

    unit_size = 3 + bool(status & 0x10) + bool(status & 0x20) + bool(status & 0x40)
    block_size = 2 + laser_count * unit_size
    optional_size = 4  # CRC following the block data
    optional_size += 17 if status & 0x04 else 0  # Function-safety data
    optional_size += 26  # Hesai packet tail
    optional_size += 4 if status & 0x01 else 0  # Sequence number
    optional_size += 22 if status & 0x02 else 0  # IMU data
    optional_size += 4  # Packet CRC
    optional_size += 32 if status & 0x08 else 0  # Cyber-security signature
    expected_size = 12 + block_count * block_size + optional_size
    if len(payload) != expected_size:
        raise ValueError(f"Unexpected Hesai v1.4 payload size {len(payload)}; expected {expected_size}.")

    distance_unit_m = distance_unit_mm / 1000.0
    decoded_blocks = []
    block_offset = 12
    for _ in range(block_count):
        azimuth_deg = int.from_bytes(payload[block_offset:block_offset + 2], "little") / 100.0
        block_offset += 2
        azimuths = np.radians((azimuth_deg + AZIMUTH_OFFSETS_DEG) % 360.0)
        elevations = np.radians(VERTICAL_ANGLES_DEG)
        distances = np.empty(laser_count, dtype=float)

        for channel in range(laser_count):
            record = payload[block_offset:block_offset + unit_size]
            raw_distance = struct.unpack_from("<H", record)[0]
            distances[channel] = raw_distance * distance_unit_m
            block_offset += unit_size

        horizontal = distances * np.cos(elevations)
        points = np.column_stack((
            horizontal * np.sin(azimuths),
            horizontal * np.cos(azimuths),
            distances * np.sin(elevations),
        ))
        points = points[distances > 0]
        decoded_blocks.append((azimuth_deg, points))

    return decoded_blocks


frames = []
frame_blocks = []
previous_azimuth = None

with PCAP_PATH.open("rb") as capture_file:
    reader = dpkt.pcap.Reader(capture_file)
    if reader.datalink() != dpkt.pcap.DLT_EN10MB:
        raise ValueError("Expected an Ethernet .pcap capture.")

    for _, raw_packet in reader:
        try:
            ethernet = dpkt.ethernet.Ethernet(raw_packet)
            network = ethernet.data
            if isinstance(network, dpkt.ip.IP):
                transport = network.data
            elif isinstance(network, dpkt.ip6.IP6):
                transport = network.data
            else:
                continue
            if not isinstance(transport, dpkt.udp.UDP):
                continue
            decoded_blocks = decode_packet(bytes(transport.data))
        except (AttributeError, dpkt.dpkt.NeedData, ValueError):
            continue

        for azimuth, points in decoded_blocks:
            if previous_azimuth is not None and previous_azimuth - azimuth > 180:
                if frame_blocks:
                    frames.append(np.vstack(frame_blocks))
                    frame_blocks = []
                if len(frames) >= MAX_FRAMES:
                    break
            if len(points):
                frame_blocks.append(points)
            previous_azimuth = azimuth

        if len(frames) >= MAX_FRAMES:
            break

if len(frames) < MAX_FRAMES and frame_blocks:
    frames.append(np.vstack(frame_blocks))
if not frames:
    raise ValueError("No valid Hesai protocol v1.4 LiDAR frames found in the capture.")

first_frame = frames[0]
print(f"Loaded {len(frames)} frame(s) (maximum {MAX_FRAMES}) from {PCAP_PATH.resolve()}")
print("Points per frame: " + ", ".join(f"{len(frame):,}" for frame in frames))

Loaded 40 frame(s) (maximum 40) from C:\Users\danag\Documents\LIDAR\data\myself in my room dancing.pcap
Points per frame: 41,628, 212,466, 211,811, 210,972, 210,335, 209,876, 208,753, 207,858, 207,285, 206,456, 206,094, 206,493, 205,931, 205,823, 205,735, 204,971, 201,732, 198,349, 199,270, 198,899, 202,886, 205,352, 205,981, 206,216, 208,889, 212,438, 213,687, 214,075, 213,973, 214,153, 214,363, 214,412, 214,507, 215,006, 214,956, 214,709, 214,756, 214,733, 214,787, 214,808


## Plot frames in sequence

In [7]:
from IPython.display import clear_output, display
import ipywidgets as widgets

PLOT_MIN_DISTANCE_M = 0.0
PLOT_MAX_DISTANCE_M = 3.0
PLOT_KEEP_FRACTION = 0.8
PLOT_RANDOM_SEED = 42


class ConstantVelocityKalman3D:
    def __init__(self, position, measurement_noise_m, process_noise):
        self.state = np.concatenate((np.asarray(position, dtype=float), np.zeros(3)))
        self.covariance = np.diag([measurement_noise_m ** 2] * 3 + [1.0] * 3)
        self.measurement_covariance = np.eye(3) * measurement_noise_m ** 2
        self.process_noise = process_noise

    def predict(self, dt=1.0):
        transition = np.eye(6)
        transition[:3, 3:] = np.eye(3) * dt
        process_covariance = np.zeros((6, 6))
        for axis in range(3):
            process_covariance[axis, axis] = self.process_noise * dt ** 4 / 4
            process_covariance[axis, axis + 3] = self.process_noise * dt ** 3 / 2
            process_covariance[axis + 3, axis] = self.process_noise * dt ** 3 / 2
            process_covariance[axis + 3, axis + 3] = self.process_noise * dt ** 2
        self.state = transition @ self.state
        self.covariance = transition @ self.covariance @ transition.T + process_covariance
        return self.state[:3].copy()

    def update(self, measurement):
        observation = np.zeros((3, 6))
        observation[:, :3] = np.eye(3)
        innovation = np.asarray(measurement, dtype=float) - observation @ self.state
        innovation_covariance = (
            observation @ self.covariance @ observation.T + self.measurement_covariance
        )
        kalman_gain = self.covariance @ observation.T @ np.linalg.inv(innovation_covariance)
        self.state = self.state + kalman_gain @ innovation
        residual = np.eye(6) - kalman_gain @ observation
        self.covariance = (
            residual @ self.covariance @ residual.T
            + kalman_gain @ self.measurement_covariance @ kalman_gain.T
        )
        return self.state[:3].copy()


method_widget = widgets.Dropdown(
    options=["K-means", "DBSCAN", "HDBSCAN", "Spectral"],
    value="DBSCAN",
    description="Method",
)
frame_count_widget = widgets.IntSlider(
    value=min(MAX_FRAMES, len(frames)),
    min=1,
    max=len(frames),
    step=1,
    description="Frames",
    continuous_update=False,
)
min_distance_widget = widgets.FloatText(value=PLOT_MIN_DISTANCE_M, description="Min range (m)")
max_distance_widget = widgets.FloatText(value=PLOT_MAX_DISTANCE_M, description="Max range (m)")
keep_fraction_widget = widgets.FloatSlider(
    value=PLOT_KEEP_FRACTION,
    min=0.05,
    max=1.0,
    step=0.05,
    description="Point fraction",
    readout_format=".2f",
    continuous_update=False,
)
cluster_count_widget = widgets.IntSlider(
    value=5,
    min=2,
    max=30,
    step=1,
    description="K-means/Spectral clusters",
    continuous_update=False,
)
dbscan_eps_widget = widgets.FloatText(value=0.1, description="DBSCAN eps (m)")
min_samples_widget = widgets.IntSlider(
    value=10,
    min=1,
    max=100,
    step=1,
    description="Min samples",
    continuous_update=False,
)
hdbscan_min_cluster_size_widget = widgets.IntSlider(
    value=10,
    min=2,
    max=100,
    description="HDBSCAN min cluster size",
    continuous_update=False,
)
spectral_neighbors_widget = widgets.IntSlider(
    value=10,
    min=1,
    max=100,
    step=1,
    description="Spectral neighbors",
    continuous_update=False,
)
random_seed_widget = widgets.IntText(value=PLOT_RANDOM_SEED, description="Sample seed")
tracking_measurement_noise_widget = widgets.FloatText(
    value=0.15, description="Measurement noise (m)"
)
tracking_process_noise_widget = widgets.FloatText(
    value=0.05, description="Motion noise (m/frame^2)"
)
tracking_gate_widget = widgets.FloatText(value=0.75, description="Association gate (m)")

scene_minimum = np.min([frame.min(axis=0) for frame in frames], axis=0)
scene_maximum = np.max([frame.max(axis=0) for frame in frames], axis=0)
bbox_xmin_widget = widgets.FloatText(value=float(scene_minimum[0]), description="X min (m)")
bbox_xmax_widget = widgets.FloatText(value=float(scene_maximum[0]), description="X max (m)")
bbox_ymin_widget = widgets.FloatText(value=float(scene_minimum[1]), description="Y min (m)")
bbox_ymax_widget = widgets.FloatText(value=float(scene_maximum[1]), description="Y max (m)")
bbox_zmin_widget = widgets.FloatText(value=float(scene_minimum[2]), description="Z min (m)")
bbox_zmax_widget = widgets.FloatText(value=float(scene_maximum[2]), description="Z max (m)")

run_button = widgets.Button(description="Cluster and track", button_style="primary", icon="play")
plot_output = widgets.Output()

controls = widgets.VBox([
    widgets.HBox([method_widget, frame_count_widget]),
    widgets.HBox([min_distance_widget, max_distance_widget, keep_fraction_widget]),
    widgets.HBox([cluster_count_widget, dbscan_eps_widget, min_samples_widget]),
    widgets.HBox([hdbscan_min_cluster_size_widget, spectral_neighbors_widget, random_seed_widget]),
    widgets.HTML("<b>Kalman tracking</b>"),
    widgets.HBox([
        tracking_measurement_noise_widget,
        tracking_process_noise_widget,
        tracking_gate_widget,
    ]),
    widgets.HTML("<b>Manual initialization box</b>"),
    widgets.HBox([bbox_xmin_widget, bbox_xmax_widget]),
    widgets.HBox([bbox_ymin_widget, bbox_ymax_widget]),
    widgets.HBox([bbox_zmin_widget, bbox_zmax_widget]),
])


def box_mask(points, box):
    lower = np.array([box["xmin"], box["ymin"], box["zmin"]])
    upper = np.array([box["xmax"], box["ymax"], box["zmax"]])
    return np.all((points >= lower) & (points <= upper), axis=1)


def best_cluster_for_box(labels, manual_mask):
    candidates = [label for label in np.unique(labels) if label != -1]
    if not candidates:
        candidates = list(np.unique(labels))

    best = None
    for label in candidates:
        predicted = labels == label
        true_positive = int(np.count_nonzero(predicted & manual_mask))
        false_positive = int(np.count_nonzero(predicted & ~manual_mask))
        false_negative = int(np.count_nonzero(~predicted & manual_mask))
        precision = true_positive / max(true_positive + false_positive, 1)
        recall = true_positive / max(true_positive + false_negative, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-12)
        candidate = {
            "label": label,
            "mask": predicted,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }
        if best is None or candidate["f1"] > best["f1"]:
            best = candidate
    return best


def cluster_centroids(points, labels):
    return {
        label: points[labels == label].mean(axis=0)
        for label in np.unique(labels)
        if label != -1 and np.any(labels == label)
    }


def draw_box_edges(axes, box):
    x_values = [box["xmin"], box["xmax"]]
    y_values = [box["ymin"], box["ymax"]]
    z_values = [box["zmin"], box["zmax"]]
    for x_value in x_values:
        for y_value in y_values:
            axes.plot([x_value, x_value], [y_value, y_value], z_values, color="blue", linewidth=1)
    for x_value in x_values:
        for z_value in z_values:
            axes.plot([x_value, x_value], y_values, [z_value, z_value], color="blue", linewidth=1)
    for y_value in y_values:
        for z_value in z_values:
            axes.plot(x_values, [y_value, y_value], [z_value, z_value], color="blue", linewidth=1)


def plot_selected_frames(_):
    run_button.disabled = True
    try:
        box = {
            "xmin": bbox_xmin_widget.value,
            "xmax": bbox_xmax_widget.value,
            "ymin": bbox_ymin_widget.value,
            "ymax": bbox_ymax_widget.value,
            "zmin": bbox_zmin_widget.value,
            "zmax": bbox_zmax_widget.value,
        }
        if any(box[f"{axis}min"] > box[f"{axis}max"] for axis in ("x", "y", "z")):
            raise ValueError("Each bounding-box minimum must be less than or equal to its maximum.")
        measurement_noise = tracking_measurement_noise_widget.value
        process_noise = tracking_process_noise_widget.value
        association_gate = tracking_gate_widget.value
        if measurement_noise <= 0 or process_noise < 0 or association_gate <= 0:
            raise ValueError("Measurement noise and association gate must be positive; motion noise cannot be negative.")

        with plot_output:
            clear_output(wait=True)
            selected_frames = frames[:frame_count_widget.value]
            frame_numbers = []
            frame_f1_scores = []
            track_frames = []
            track_positions = []
            detection_positions = []
            tracker = None

            for frame_number, frame in enumerate(selected_frames, start=1):
                plot_points = prepare_points(
                    frame,
                    min_distance=min_distance_widget.value,
                    max_distance=max_distance_widget.value,
                    keep_fraction=keep_fraction_widget.value,
                    seed=random_seed_widget.value + frame_number - 1,
                )
                predicted_position = tracker.predict() if tracker is not None else None
                if not len(plot_points):
                    frame_numbers.append(frame_number)
                    frame_f1_scores.append(0.0)
                    track_frames.append(frame_number)
                    track_positions.append(
                        tracker.state[:3].copy() if tracker is not None else np.full(3, np.nan)
                    )
                    detection_positions.append(np.full(3, np.nan))
                    print(f"Frame {frame_number}/{len(selected_frames)}: no points remain after filtering; F1 = 0.000; track predicted")
                    continue

                labels = cluster_points(
                    plot_points,
                    method_widget.value,
                    cluster_count_widget.value,
                    dbscan_eps_widget.value,
                    min_samples_widget.value,
                    hdbscan_min_cluster_size_widget.value,
                    spectral_neighbors_widget.value,
                )
                manual_mask = box_mask(plot_points, box)
                best = best_cluster_for_box(labels, manual_mask)
                centroids = cluster_centroids(plot_points, labels)
                detection = None
                detection_label = None

                if tracker is None:
                    if best is not None and np.any(best["mask"] & manual_mask):
                        detection_label = best["label"]
                        detection = centroids.get(detection_label)
                elif centroids:
                    detection_label = min(
                        centroids,
                        key=lambda label: np.linalg.norm(centroids[label] - predicted_position),
                    )
                    candidate_position = centroids[detection_label]
                    if np.linalg.norm(candidate_position - predicted_position) <= association_gate:
                        detection = candidate_position
                    else:
                        detection_label = None

                if tracker is None and detection is not None:
                    tracker = ConstantVelocityKalman3D(
                        detection, measurement_noise, process_noise
                    )
                    track_status = "initialized"
                elif tracker is not None and detection is not None:
                    tracker.update(detection)
                    track_status = "updated"
                elif tracker is not None:
                    track_status = "predicted (no associated detection)"
                else:
                    track_status = "waiting for box-matched detection"

                frame_numbers.append(frame_number)
                frame_f1 = best["f1"] if best is not None else 0.0
                frame_f1_scores.append(frame_f1)
                track_frames.append(frame_number)
                track_positions.append(
                    tracker.state[:3].copy() if tracker is not None else np.full(3, np.nan)
                )
                detection_positions.append(
                    detection.copy() if detection is not None else np.full(3, np.nan)
                )
                cluster_count = len(set(labels.tolist()) - {-1})
                noise_count = int(np.count_nonzero(labels == -1))
                metric_text = (
                    f"best box label {best['label']}, precision {best['precision']:.3f}, "
                    f"recall {best['recall']:.3f}, F1 {frame_f1:.3f}"
                    if best is not None else "no cluster labels, F1 0.000"
                )
                tracking_text = (
                    f"track {track_status}"
                    + (f" using label {detection_label}" if detection_label is not None else "")
                    + (f" at [{tracker.state[0]:.2f}, {tracker.state[1]:.2f}, {tracker.state[2]:.2f}] m" if tracker is not None else "")
                )
                print(
                    f"Frame {frame_number}/{len(selected_frames)}: {cluster_count} clusters, "
                    f"{noise_count:,} noise points; {metric_text}; {tracking_text}"
                )

                if best is None:
                    continue
                true_positive = best["mask"] & manual_mask
                false_positive = best["mask"] & ~manual_mask
                false_negative = ~best["mask"] & manual_mask
                background = ~best["mask"] & ~manual_mask

                figure = plt.figure(figsize=(10, 8))
                axes = figure.add_subplot(111, projection="3d")
                axes.scatter(*plot_points[background].T, c="lightgray", s=1, alpha=0.25)
                axes.scatter(*plot_points[false_negative].T, c="blue", s=4, label="Box points missed")
                axes.scatter(*plot_points[false_positive].T, c="red", s=4, label="Cluster outside box")
                axes.scatter(*plot_points[true_positive].T, c="green", s=4, label="Cluster inside box")
                draw_box_edges(axes, box)
                if tracker is not None:
                    axes.scatter(
                        *tracker.state[:3], c="magenta", marker="*", s=140,
                        label="Kalman track position",
                    )
                axes.set_title(
                    f"Hesai JT128 frame {frame_number} ({method_widget.value}), "
                    f"best label {best['label']}, F1 {frame_f1:.3f}"
                )
                axes.set_xlabel("X (m)")
                axes.set_ylabel("Y (m)")
                axes.set_zlabel("Z (m)")
                axes.set_box_aspect(np.maximum(np.ptp(plot_points, axis=0), 1e-6))
                axes.legend(loc="upper right")
                plt.tight_layout()
                plt.show()
                plt.close(figure)

            valid_track = np.all(np.isfinite(track_positions), axis=1)
            if np.any(valid_track):
                track_array = np.asarray(track_positions)[valid_track]
                track_frame_array = np.asarray(track_frames)[valid_track]
                detections_array = np.asarray(detection_positions)[valid_track]
                figure = plt.figure(figsize=(9, 7))
                axes = figure.add_subplot(111, projection="3d")
                axes.plot(*track_array.T, color="magenta", marker="o", label="Kalman estimate")
                valid_detections = np.all(np.isfinite(detections_array), axis=1)
                if np.any(valid_detections):
                    axes.scatter(
                        *detections_array[valid_detections].T,
                        c="green", marker="x", s=45, label="Associated cluster detections",
                    )
                for position, frame_number in zip(track_array, track_frame_array):
                    axes.text(*position, str(frame_number), fontsize=8)
                axes.set_title("Kalman object track across frames")
                axes.set_xlabel("X (m)")
                axes.set_ylabel("Y (m)")
                axes.set_zlabel("Z (m)")
                axes.legend(loc="upper right")
                plt.tight_layout()
                plt.show()
                plt.close(figure)
            else:
                print("No box-matched object detection was available to initialize the track.")

            figure, axes = plt.subplots(figsize=(10, 4))
            axes.plot(frame_numbers, frame_f1_scores, marker="o", linewidth=1.5)
            axes.set_title(f"Best cluster F1 by frame ({method_widget.value})")
            axes.set_xlabel("Frame number")
            axes.set_ylabel("F1 score")
            axes.set_ylim(0, 1)
            axes.set_xticks(frame_numbers)
            axes.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
            plt.close(figure)
    except Exception as error:
        with plot_output:
            print(f"Clustering failed: {error}")
    finally:
        run_button.disabled = False


run_button.on_click(plot_selected_frames)
display(controls, run_button, plot_output)

Button(button_style='primary', description='Cluster and track', icon='play', style=ButtonStyle())

Output()

In [ ]:
%matplotlib widget

from IPython.display import clear_output, display
import ipywidgets as widgets

single_frame_widget = widgets.IntSlider(
    value=1,
    min=1,
    max=len(frames),
    step=1,
    description="Frame",
    continuous_update=False,
)
single_frame_button = widgets.Button(description="Show frame", button_style="primary", icon="eye")
single_frame_output = widgets.Output()
SINGLE_FRAME_POINT_LIMIT = 30000


def plot_single_frame(_):
    single_frame_button.disabled = True
    try:
        frame_number = single_frame_widget.value
        frame = frames[frame_number - 1]
        points_to_plot = prepare_points(
            frame,
            min_distance=min_distance_widget.value,
            max_distance=max_distance_widget.value,
            keep_fraction=keep_fraction_widget.value,
            seed=random_seed_widget.value + frame_number - 1,
        )
        if not len(points_to_plot):
            raise ValueError("No points remain after applying the current distance and sampling settings.")

        box = {
            "xmin": bbox_xmin_widget.value,
            "xmax": bbox_xmax_widget.value,
            "ymin": bbox_ymin_widget.value,
            "ymax": bbox_ymax_widget.value,
            "zmin": bbox_zmin_widget.value,
            "zmax": bbox_zmax_widget.value,
        }
        if any(box[f"{axis}min"] > box[f"{axis}max"] for axis in ("x", "y", "z")):
            raise ValueError("Each bounding-box minimum must be less than or equal to its maximum.")

        inside_box = box_mask(points_to_plot, box)
        inside_points = points_to_plot[inside_box]
        outside_points = points_to_plot[~inside_box]
        rng = np.random.default_rng(random_seed_widget.value + frame_number - 1)
        if len(inside_points) >= SINGLE_FRAME_POINT_LIMIT:
            inside_points = inside_points[rng.choice(len(inside_points), SINGLE_FRAME_POINT_LIMIT, replace=False)]
            outside_points = np.empty((0, 3))
        else:
            outside_limit = SINGLE_FRAME_POINT_LIMIT - len(inside_points)
            if len(outside_points) > outside_limit:
                outside_points = outside_points[rng.choice(len(outside_points), outside_limit, replace=False)]

        with single_frame_output:
            clear_output(wait=True)
            figure = plt.figure(figsize=(10, 8))
            axes = figure.add_subplot(111, projection="3d")
            if len(outside_points):
                axes.scatter(*outside_points.T, c="lightgray", s=1, alpha=0.3, label="Outside box")
            if len(inside_points):
                axes.scatter(*inside_points.T, c="crimson", s=3, alpha=0.9, label="Inside box")
            draw_box_edges(axes, box)
            axes.set_title(f"Frame {frame_number}: {len(points_to_plot):,} points, {int(inside_box.sum()):,} inside box")
            axes.set_xlabel("X (m)")
            axes.set_ylabel("Y (m)")
            axes.set_zlabel("Z (m)")
            axes.set_box_aspect(np.maximum(np.ptp(points_to_plot, axis=0), 1e-6))
            axes.legend(loc="upper right")
            plt.tight_layout()
            plt.show()
    except Exception as error:
        with single_frame_output:
            clear_output(wait=True)
            print(f"Frame preview failed: {error}")
    finally:
        single_frame_button.disabled = False


single_frame_button.on_click(plot_single_frame)
plot_single_frame(None)
display(widgets.HBox([single_frame_widget, single_frame_button]), single_frame_output)

Output()